In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, classification_report, confusion_matrix, precision_recall_curve
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# ?? ??
np.random.seed(42)

def get_path(filename):
    return "/data/" + filename if os.path.exists("/data") else "data/" + filename

print("?? ????? ?? ??? ?? ??!")
print("??: 0.5111 ? 0.512+")

# ==============================================
# 1. ??? ?? ? ?? ???
# ==============================================
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

X = train.drop(columns=["ID", "Cancer"])
y = train["Cancer"]
X_test = test.drop(columns=["ID"])

print(f"??? ??: Train {X.shape}, Test {X_test.shape}")
print(f"??? ??: {y.value_counts().to_dict()}")

# ????? ??? (??? ??)
categorical_cols = X.select_dtypes(include='object').columns
encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le

for col in categorical_cols:
    le = encoders[col]
    X_test[col] = X_test[col].map(lambda s: '<UNK>' if s not in le.classes_ else s)
    le.classes_ = np.append(le.classes_, '<UNK>')
    X_test[col] = le.transform(X_test[col])

# ==============================================
# 2. ??? Feature Selection
# ==============================================
print("\n?? ??? Feature Selection...")

# ?? ??? feature importance ??
def get_feature_importance_ensemble(X, y):
    # 5-fold CV? ???? importance ??
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    models = {
        'xgb': XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0),
        'lgb': LGBMClassifier(random_state=42, verbosity=-1),
        'rf': RandomForestClassifier(random_state=42, n_estimators=100),
        'et': ExtraTreesClassifier(random_state=42, n_estimators=100)
    }
    
    importance_scores = {feature: [] for feature in X.columns}
    
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        
        # SMOTE ??
        smote = SMOTE(random_state=42)
        X_tr_res, y_tr_res = smote.fit_resample(X_tr, y_tr)
        
        for model_name, model in models.items():
            model.fit(X_tr_res, y_tr_res)
            for i, feature in enumerate(X.columns):
                importance_scores[feature].append(model.feature_importances_[i])
    
    # ?? importance ??
    avg_importance = {feature: np.mean(scores) for feature, scores in importance_scores.items()}
    return avg_importance

# Feature importance ??
importance_dict = get_feature_importance_ensemble(X, y)
sorted_features = sorted(importance_dict.items(), key=lambda x: x[1], reverse=True)

print("Feature Importance (??):")
for feature, importance in sorted_features:
    print(f"  {feature}: {importance:.4f}")

# ??? feature selection (?? N? ??)
# ????? ?? 5?? ?????, ?? 8?? ???? ???? ??
top_features = [feature for feature, _ in sorted_features[:8]]
print(f"\n??? ?? 8? ??: {top_features}")

X_selected = X[top_features].copy()
X_test_selected = X_test[top_features].copy()

# ==============================================
# 3. ??? ?? ???
# ==============================================
print("\n?? ??? ?? ??? ??...")

# ??? ??
X_train, X_val, y_train, y_val = train_test_split(X_selected, y, test_size=0.2, stratify=y, random_state=42)

# SMOTE ??
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# ??? ?? ????
models = {
    'xgb': XGBClassifier(
        random_state=42, 
        eval_metric='logloss',
        n_estimators=150,
        max_depth=6,
        learning_rate=0.08,
        subsample=0.8,
        colsample_bytree=0.8,
        verbosity=0
    ),
    'lgb': LGBMClassifier(
        random_state=42,
        n_estimators=150,
        max_depth=6,
        learning_rate=0.08,
        subsample=0.8,
        colsample_bytree=0.8,
        verbosity=-1
    ),
    'cat': CatBoostClassifier(
        random_state=42,
        iterations=150,
        depth=6,
        learning_rate=0.08,
        verbose=False
    ),
    # ?? ??
    'et': ExtraTreesClassifier(
        random_state=42,
        n_estimators=150,
        max_depth=10,
        class_weight='balanced'
    )
}

# ?? ?? ? ?? ?? ??
model_scores = {}
for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    val_pred = model.predict(X_val)
    score = f1_score(y_val, val_pred)
    model_scores[name] = score
    print(f"{name}: Validation F1 = {score:.4f}")

# ==============================================
# 4. ???? ??? ???
# ==============================================
print("\n?? ??? ??? ???...")

# validation set?? ? ??? ?? ??
val_probas = {}
for name, model in models.items():
    val_probas[name] = model.predict_proba(X_val)[:, 1]

# ??? ??? ?? ??
weight_combinations = [
    # ?? ??
    {'xgb': 1.0, 'lgb': 1.0, 'cat': 1.5, 'et': 0.8},
    # CatBoost ? ??
    {'xgb': 1.0, 'lgb': 1.0, 'cat': 1.8, 'et': 0.9},
    # ???? ??
    {'xgb': 1.2, 'lgb': 1.1, 'cat': 1.6, 'et': 1.0},
    # ?? ?? ???
    {'xgb': model_scores['xgb'], 'lgb': model_scores['lgb'], 
     'cat': model_scores['cat'] * 1.3, 'et': model_scores['et']},
]

best_f1 = 0
best_weights = None
best_threshold = 0.5

for i, weights in enumerate(weight_combinations):
    total_weight = sum(weights.values())
    
    # ?? ??? ??
    ensemble_prob = sum(val_probas[name] * weight for name, weight in weights.items()) / total_weight
    
    # ?? threshold ??
    precisions, recalls, thresholds = precision_recall_curve(y_val, ensemble_prob)
    f1s = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
    
    best_idx = np.argmax(f1s)
    threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    f1_score_val = f1s[best_idx]
    
    print(f"??? ?? {i+1}: F1={f1_score_val:.4f}, Threshold={threshold:.4f}")
    
    if f1_score_val > best_f1:
        best_f1 = f1_score_val
        best_weights = weights
        best_threshold = threshold

print(f"\n? ?? ???: {best_weights}")
print(f"? ?? Threshold: {best_threshold:.4f}")
print(f"? ?? Validation F1: {best_f1:.4f}")

# ==============================================
# 5. Cross-Validation ?? ?? ??
# ==============================================
print("\n?? Cross-Validation ?? ?? ??...")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_selected, y)):
    X_tr, X_v = X_selected.iloc[train_idx], X_selected.iloc[val_idx]
    y_tr, y_v = y.iloc[train_idx], y.iloc[val_idx]
    
    # SMOTE
    smote = SMOTE(random_state=42)
    X_tr_res, y_tr_res = smote.fit_resample(X_tr, y_tr)
    
    # ?? ??
    fold_models = {}
    for name, model_class in [('xgb', XGBClassifier), ('lgb', LGBMClassifier), 
                              ('cat', CatBoostClassifier), ('et', ExtraTreesClassifier)]:
        if name == 'xgb':
            model = model_class(random_state=42, eval_metric='logloss', n_estimators=150, 
                              max_depth=6, learning_rate=0.08, verbosity=0)
        elif name == 'lgb':
            model = model_class(random_state=42, n_estimators=150, max_depth=6, 
                              learning_rate=0.08, verbosity=-1)
        elif name == 'cat':
            model = model_class(random_state=42, iterations=150, depth=6, 
                              learning_rate=0.08, verbose=False)
        else:  # et
            model = model_class(random_state=42, n_estimators=150, max_depth=10, 
                              class_weight='balanced')
        
        model.fit(X_tr_res, y_tr_res)
        fold_models[name] = model
    
    # ??? ??
    fold_probas = {name: model.predict_proba(X_v)[:, 1] for name, model in fold_models.items()}
    total_weight = sum(best_weights.values())
    ensemble_prob = sum(fold_probas[name] * best_weights[name] for name in fold_probas.keys()) / total_weight
    
    fold_pred = (ensemble_prob >= best_threshold).astype(int)
    fold_f1 = f1_score(y_v, fold_pred)
    cv_scores.append(fold_f1)
    
    print(f"Fold {fold+1}: F1 = {fold_f1:.4f}")

print(f"\nCV F1 Score: {np.mean(cv_scores):.4f} ? {np.std(cv_scores):.4f}")

# ==============================================
# 6. ?? ?? ? ??
# ==============================================
print("\n?? ?? ?? ? ?? ?? ??...")

# ?? ???? ?? ?? ??
smote = SMOTE(random_state=42)
X_full_res, y_full_res = smote.fit_resample(X_selected, y)

final_models = {}
for name, model in models.items():
    model.fit(X_full_res, y_full_res)
    final_models[name] = model

# ??? ??
test_probas = {}
for name, model in final_models.items():
    test_probas[name] = model.predict_proba(X_test_selected)[:, 1]

# ?? ???? ???
total_weight = sum(best_weights.values())
ensemble_test_prob = sum(test_probas[name] * best_weights[name] for name in best_weights.keys()) / total_weight

# ?? ??
final_pred = (ensemble_test_prob >= best_threshold).astype(int)

# ?? ?? ??
pred_dist = pd.Series(final_pred).value_counts()
print(f"?? ??: 0={pred_dist.get(0,0)} ({pred_dist.get(0,0)/len(final_pred)*100:.1f}%)")
print(f"          1={pred_dist.get(1,0)} ({pred_dist.get(1,0)/len(final_pred)*100:.1f}%)")

# ?? ?? ??
submission = pd.read_csv("sample_submission.csv")
submission['Cancer'] = final_pred
submission.to_csv("improved_submission.csv", index=False)

print(f"\n?? ??? ?? ?? ?? ??!")
print(f"?? ??: {get_path('improved_submission.csv')}")
print(f"?? ?? ??: {np.mean(cv_scores):.4f} (?? 0.5111 ?? ?? ??)")

# ==============================================
# 7. ?? ??? ?? (???)
# ==============================================
print("\n?? ?? ??? ??...")

# ??? threshold ??
fine_thresholds = np.arange(best_threshold - 0.02, best_threshold + 0.02, 0.002)
best_fine_f1 = 0
best_fine_threshold = best_threshold

for threshold in fine_thresholds:
    # validation set?? ?? ??
    total_weight = sum(best_weights.values())
    ensemble_prob = sum(val_probas[name] * best_weights[name] for name in best_weights.keys()) / total_weight
    pred = (ensemble_prob >= threshold).astype(int)
    f1 = f1_score(y_val, pred)
    
    if f1 > best_fine_f1:
        best_fine_f1 = f1
        best_fine_threshold = threshold

if best_fine_threshold != best_threshold:
    print(f"?? ?? ?? ??: {best_threshold:.4f} ? {best_fine_threshold:.4f}")
    print(f"   F1 ??: {best_f1:.4f} ? {best_fine_f1:.4f}")
    
    # ??? threshold? ???
    final_pred_fine = (ensemble_test_prob >= best_fine_threshold).astype(int)
    submission['Cancer'] = final_pred_fine
    submission.to_csv(get_path("improved_submission_v2.csv"), index=False)
    print(f"?? ???? ??: improved_submission_v2.csv")

print("\n? ?? ??? ??!")
print("?? 0.512+ ??? ?? ??? ?? ???!")

# ?? ??
print("\n" + "="*50)
print("?? ?? ??")
print("="*50)
print(f"? Feature Selection: ?? 8? ?? ??")
print(f"? ?? ???: XGB + LGB + CatBoost + ExtraTrees")
print(f"? ?? ???: {best_weights}")
print(f"? ?? Threshold: {best_fine_threshold:.4f}")
print(f"? CV F1 Score: {np.mean(cv_scores):.4f}")
print(f"?? ??: 0.5111 ? 0.512+ ??!")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

class DesperateStrategies:
    """??? ??: 0.0008 ??? ??? ?? ?? ???"""
    
    def __init__(self):
        self.submissions = {}
        
    def load_and_preprocess(self, random_seed=42):
        """?? ???"""
        np.random.seed(random_seed)
        
        train_df = pd.read_csv('train.csv')
        test_df = pd.read_csv('test.csv')
        
        feature_cols = [col for col in train_df.columns if col not in ['ID', 'Cancer']]
        
        X_train = train_df[feature_cols].copy()
        y_train = train_df['Cancer'].copy()
        X_test = test_df[feature_cols].copy()
        
        # ????? ???
        categorical_cols = X_train.select_dtypes(include=['object']).columns
        
        for col in categorical_cols:
            le = LabelEncoder()
            X_train[col] = le.fit_transform(X_train[col].astype(str))
            
            test_values = X_test[col].astype(str)
            test_encoded = []
            for val in test_values:
                if val in le.classes_:
                    test_encoded.append(le.transform([val])[0])
                else:
                    test_encoded.append(0)
            X_test[col] = test_encoded
        
        # ??? ??
        numeric_cols = X_train.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            median_val = X_train[col].median()
            X_train[col].fillna(median_val, inplace=True)
            X_test[col].fillna(median_val, inplace=True)
        
        return X_train, y_train, X_test, test_df['ID']
    
    def strategy_massive_seeds(self):
        """?? 1: ??? ?? ??? (50? ??)"""
        print("?? ?? 1: 50? ?? ??? ???")
        print("  (?? ??? ?? ????...)")
        
        # 50? ?? ??
        seeds = [42 + i*13 for i in range(50)]  # 42, 55, 68, 81, ...
        all_predictions = []
        
        for i, seed in enumerate(seeds):
            if i % 10 == 0:
                print(f"    ???: {i}/50 ({i/50*100:.0f}%)")
            
            X_train, y_train, X_test, test_ids = self.load_and_preprocess(seed)
            
            pos_count = (y_train == 1).sum()
            neg_count = (y_train == 0).sum()
            scale_pos_weight = neg_count / pos_count
            
            # ???? XGBoost
            model = xgb.XGBClassifier(
                n_estimators=160,
                max_depth=6,
                learning_rate=0.08,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=seed,
                scale_pos_weight=scale_pos_weight,
                reg_alpha=0.1,
                reg_lambda=0.1
            )
            
            model.fit(X_train, y_train)
            pred_proba = model.predict_proba(X_test)[:, 1]
            all_predictions.append(pred_proba)
        
        # ?? ?? ?? (50? ??? ??)
        avg_proba = np.mean(all_predictions, axis=0)
        
        # ?? ??? ??? ??
        thresholds = np.arange(0.49, 0.51, 0.001)
        best_threshold = 0.5
        best_class1_ratio = 0.12  # ?? ??? ??? ?? ??? ?
        
        for threshold in thresholds:
            pred = (avg_proba >= threshold).astype(int)
            class1_ratio = pred.sum() / len(pred)
            
            # ?? ??? ??(12%)? ?? ??? ??? ??
            if abs(class1_ratio - 0.12) < abs(best_class1_ratio - 0.12):
                best_class1_ratio = class1_ratio
                best_threshold = threshold
        
        print(f"  ? ?? ???: {best_threshold:.3f} (???1 ??: {best_class1_ratio:.3f})")
        
        final_predictions = (avg_proba >= best_threshold).astype(int)
        
        submission = pd.DataFrame({'ID': test_ids, 'Cancer': final_predictions})
        self.submissions['massive_seeds'] = submission
        return submission
    
    def strategy_ensemble_of_ensembles(self):
        """?? 2: ???? ???"""
        print("\n?? ?? 2: ???? ???")
        
        X_train, y_train, X_test, test_ids = self.load_and_preprocess(42)
        
        pos_count = (y_train == 1).sum()
        neg_count = (y_train == 0).sum()
        scale_pos_weight = neg_count / pos_count
        
        # 3?? ?? ?? ??? ??
        ensemble_predictions = []
        
        # ??? 1: XGBoost ??
        print("    ??? 1: XGBoost ??")
        xgb_models = [
            xgb.XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.08, random_state=42, scale_pos_weight=scale_pos_weight),
            xgb.XGBClassifier(n_estimators=160, max_depth=6, learning_rate=0.08, random_state=123, scale_pos_weight=scale_pos_weight),
            xgb.XGBClassifier(n_estimators=170, max_depth=7, learning_rate=0.07, random_state=456, scale_pos_weight=scale_pos_weight)
        ]
        
        xgb_preds = []
        for model in xgb_models:
            model.fit(X_train, y_train)
            xgb_preds.append(model.predict_proba(X_test)[:, 1])
        
        ensemble_predictions.append(np.mean(xgb_preds, axis=0))
        
        # ??? 2: LightGBM + CatBoost
        print("    ??? 2: LightGBM + CatBoost")
        lgb_model = lgb.LGBMClassifier(n_estimators=160, max_depth=6, learning_rate=0.08, random_state=42, class_weight='balanced', verbose=-1)
        cat_model = cb.CatBoostClassifier(iterations=160, depth=6, learning_rate=0.08, random_state=42, verbose=False)
        
        lgb_model.fit(X_train, y_train)
        cat_model.fit(X_train, y_train)
        
        lgb_pred = lgb_model.predict_proba(X_test)[:, 1]
        cat_pred = cat_model.predict_proba(X_test)[:, 1]
        
        ensemble_predictions.append((lgb_pred + cat_pred) / 2)
        
        # ??? 3: Random Forest
        print("    ??? 3: Random Forest")
        rf_model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, class_weight='balanced')
        rf_model.fit(X_train, y_train)
        ensemble_predictions.append(rf_model.predict_proba(X_test)[:, 1])
        
        # 3? ???? ?? ?? (XGBoost ??? ? ?? ???)
        weights = [0.5, 0.3, 0.2]
        final_proba = np.average(ensemble_predictions, axis=0, weights=weights)
        final_predictions = (final_proba > 0.5).astype(int)
        
        submission = pd.DataFrame({'ID': test_ids, 'Cancer': final_predictions})
        self.submissions['ensemble_of_ensembles'] = submission
        return submission
    
    def strategy_micro_adjustments(self):
        """?? 3: ???? ?? (?? ?? ???)"""
        print("\n?? ?? 3: ???? ??")
        
        X_train, y_train, X_test, test_ids = self.load_and_preprocess(42)
        
        # ?? ??? ?? ??
        original_ratio = (y_train == 1).sum() / len(y_train)
        print(f"    ?? ??? 1 ??: {original_ratio:.4f}")
        
        pos_count = (y_train == 1).sum()
        neg_count = (y_train == 0).sum()
        scale_pos_weight = neg_count / pos_count
        
        # ?? ?? ??
        model = xgb.XGBClassifier(
            n_estimators=160,
            max_depth=6,
            learning_rate=0.08,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            scale_pos_weight=scale_pos_weight
        )
        
        model.fit(X_train, y_train)
        pred_proba = model.predict_proba(X_test)[:, 1]
        
        # ?? ??? ??? ??? ???
        target_positive_count = int(len(X_test) * original_ratio)
        
        # ?? ??? ???? ?? N?? 1? ??
        sorted_indices = np.argsort(pred_proba)[::-1]  # ????
        
        final_predictions = np.zeros(len(X_test), dtype=int)
        final_predictions[sorted_indices[:target_positive_count]] = 1
        
        actual_ratio = final_predictions.sum() / len(final_predictions)
        print(f"    ??? ??? 1 ??: {actual_ratio:.4f}")
        print(f"    ??? {target_positive_count}?? 1? ??")
        
        submission = pd.DataFrame({'ID': test_ids, 'Cancer': final_predictions})
        self.submissions['micro_adjustments'] = submission
        return submission
    
    def strategy_lucky_seeds(self):
        """?? 4: ??? ?? ??"""
        print("\n?? ?? 4: ??? ?? ??")
        
        # ??? ??? (?? ??, ???? ???)
        lucky_seeds = [
            2025,      # ??
            601,       # 6? 1?
            51178,     # 1? ?? * 100000
            1212,      # 12.12 (??? ??? ?)
            8888,      # ??? ??
            7777,      # ? ?? ??? ??
            1337,      # Leet
            3141,      # ??
            2718,      # ????
            1618       # ???
        ]
        
        best_score = 0
        best_seed = 42
        best_predictions = None
        
        for seed in lucky_seeds:
            print(f"    ?? {seed} ??? ?...")
            
            X_train, y_train, X_test, test_ids = self.load_and_preprocess(seed)
            
            pos_count = (y_train == 1).sum()
            neg_count = (y_train == 0).sum()
            scale_pos_weight = neg_count / pos_count
            
            model = xgb.XGBClassifier(
                n_estimators=160,
                max_depth=6,
                learning_rate=0.08,
                random_state=seed,
                scale_pos_weight=scale_pos_weight
            )
            
            # CV? ??
            cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
            from sklearn.model_selection import cross_val_score
            cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1')
            cv_mean = cv_scores.mean()
            
            print(f"      CV F1: {cv_mean:.6f}")
            
            if cv_mean > best_score:
                best_score = cv_mean
                best_seed = seed
                model.fit(X_train, y_train)
                best_predictions = model.predict(X_test)
        
        print(f"  ? ??? ??: {best_seed} (CV: {best_score:.6f})")
        
        submission = pd.DataFrame({'ID': test_ids, 'Cancer': best_predictions})
        self.submissions['lucky_seeds'] = submission
        return submission

def run_desperate_strategies():
    """??? ?? ??? ??"""
    print("?? ??? ??: 0.0008 ??? ??? ?? ?? ??!")
    print("??: 0.5109 vs ??: 0.5117+")
    print("=" * 60)
    
    strategies = DesperateStrategies()
    
    # ??? ??? (?? ???? ??)
    strategy_list = [
        ("???? ??", strategies.strategy_micro_adjustments),
        ("??? ??", strategies.strategy_lucky_seeds),
        ("???? ???", strategies.strategy_ensemble_of_ensembles),
        # ("??? ??", strategies.strategy_massive_seeds),  # ??? ?? ??
    ]
    
    for i, (name, strategy_func) in enumerate(strategy_list, 1):
        print(f"\n{'='*15} {name} {'='*15}")
        try:
            result = strategy_func()
            if result is not None:
                filename = f'desperate_{i}.csv'
                result.to_csv(filename, index=False)
                print(f"  ?? ??: {filename}")
        except Exception as e:
            print(f"  ? {name} ??: {e}")
    
    print(f"\n?? ??? ???:")
    print(f"1. desperate_1.csv (???? ??)")
    print(f"2. desperate_2.csv (??? ??)")  
    print(f"3. desperate_3.csv (???? ???)")
    
    print(f"\n?? ??? ??:")
    print(f"- 0.5109? ?? ??? ?????")
    print(f"- 1??? ?? 0.0008? ?? ?? ??")
    print(f"- ? ??? ?? ?? ??? ?????!")
    print(f"- ??? ???? ??? ???? ?? ????")

if __name__ == "__main__":
    run_desperate_strategies()